# ChessEngine: paid cloud training
This notebook runs the same project package as local training. Use a **paid Colab runtime with a positive compute balance** for chess training. Start locally and transfer only the necessary dataset and checkpoints. Keep the private repository private. Never put GitHub tokens in notebook cells.

Download a ZIP of your reviewed repository or upload an archive from your local checkout. Prefer an archive without data, `.git`, caches, or secrets. Upload labeled data/checkpoints separately. Set a maximum affordable session duration and download checkpoints before disconnecting. No purchases or instances are provisioned by this notebook.

In [ ]:
from google.colab import files
print('Upload the project ZIP, labeled JSONL, and any resume/checkpoint archive.')
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as archive:
            root = Path('/content/project').resolve()
            for member in archive.infolist():
                if not (root / member.filename).resolve().is_relative_to(root):
                    raise ValueError('Unsafe archive path')
            archive.extractall(root)
projects = list(Path('/content/project').rglob('pyproject.toml'))
assert len(projects) == 1, 'Upload one project archive'
%cd {projects[0].parent}
%pip install -e '.[train]'


## Select the run explicitly
Use the same committed JSON configuration as local training. CPU is often faster for tiny updates; CUDA may help larger batches. The teacher dataset and resume files must be uploaded to the paths below. Resume checkpoints are trusted local files only. Keep the budget in C$ and reserve 20%.

In [ ]:
import json, time
from chessengine.train import fit
from chessengine.common import read_json
DATASET = '/content/pilot-labeled.jsonl'  # Set to the uploaded labeled dataset
OUTPUT = '/content/run'
config = read_json('configs/supervised.json')
config['device'] = 'cuda'
started = time.monotonic()
# Run only after confirming the paid runtime and remaining compute balance.
print(fit(DATASET, OUTPUT, config))
print('Elapsed minutes:', (time.monotonic()-started)/60)


## Self-play
Self-play calls the same CPU search. GPU availability does not speed up move generation automatically. Begin with the pilot configuration and inspect completion/truncation rates. For the main experiment, use the frozen study configuration and repeat the agreed seeds.

In [ ]:
from chessengine.rl import selfplay
# Uncomment for the selected RL run; do not run both modes accidentally.
# config = read_json('configs/pilot-rl.json')
# print(selfplay('/content/run/best.npz', 'configs/openings.json', '/content/rl-run', config))


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/chessengine-checkpoints', 'zip', OUTPUT)
files.download('/content/chessengine-checkpoints.zip')
